# Lead Scoring - Pipeline Integrado de Pre-Producción

**Objetivo:** Pipeline final integrado: carga → transformación → modelización

**Consolidación de:** A_01 (Importación) + A_02 (Calidad) + A_04 (Preparación) + A_05 (Preselección) + A_07 (Modelización)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

print(f"✓ Librerías importadas")
print(f"✓ Directorio de trabajo: {NOTEBOOK_DIR}")

✓ Librerías importadas
✓ Directorio de trabajo: c:\Users\robin\dev\01_LEADSCORING\03_notebooks


## FASE 0 - Carga del CSV Original

In [2]:
csv_path = PROJECT_ROOT / "02_datos" / "01_Originales" / "Leads.csv"
df = pd.read_csv(str(csv_path), sep=";")
print(f"✓ CSV cargado: {len(df):,} registros × {len(df.columns)} columnas")

✓ CSV cargado: 9,093 registros × 21 columnas


## FASE 1 - Limpieza (A_01, A_02)

In [3]:
# Eliminar nulos críticos
columnas_criticas = ['visitas_total', 'paginas_vistas_visita']
mask_nulos = df[columnas_criticas].isna().any(axis=1)
df = df[~mask_nulos].reset_index(drop=True)

# Imputar scores
columnas_scores = ['score_actividad', 'score_perfil']
mask_ambos_nulos = df[columnas_scores].isna().all(axis=1)
df['usuario_nuevo'] = 0
df.loc[mask_ambos_nulos, 'usuario_nuevo'] = 1
df[columnas_scores] = df[columnas_scores].fillna(0)

# Imputar categóricas
df['fuente'] = df['fuente'].fillna('Unknown')
df['ambito'] = df['ambito'].fillna('Not Specified')
df['ocupacion'] = df['ocupacion'].fillna('Not Provided')

# Eliminar columnas sin varianza
df = df.drop(columns=['conociste_revista', 'conociste_youtube', 'no_llamar'])

print(f"✓ Limpieza completada: {len(df):,} registros × {len(df.columns)} columnas")

✓ Limpieza completada: 8,970 registros × 19 columnas


## FASE 2 - Split Train/Validation

In [4]:
train_df, validation_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df['usuario_nuevo']
)
print(f"✓ Train: {len(train_df):,} | Validation: {len(validation_df):,}")

✓ Train: 6,279 | Validation: 2,691


## FASE 3 - Preparación de Datos (A_04)

In [5]:
# Separar target y features
y = train_df['compra'].copy()
X = train_df.drop(columns=['id', 'no_enviar_email', 'compra']).copy()

print(f"✓ Target (y): {len(y)} registros")
print(f"✓ Features (X): {X.shape}")

✓ Target (y): 6279 registros
✓ Features (X): (6279, 16)


In [6]:
var_ohe = ['origen', 'fuente', 'ult_actividad', 'ambito', 'ocupacion', 'descarga_lm']
var_bin = ['conociste_google', 'conociste_periodico', 'conociste_facebook', 'conociste_referencias']
num_escalar = ['visitas_total', 'tiempo_en_site_total', 'paginas_vistas_visita', 'score_actividad', 'score_perfil']
var_sin_transform = ['usuario_nuevo']

print(f"✓ Variables de agrupación definidas para A_09")

✓ Variables de agrupación definidas para A_09


In [7]:
# Convertir binarias a números
X_bin = X[var_bin].copy()
for col in var_bin:
    X_bin[col] = (X_bin[col] == 'Yes').astype(int)

# MinMaxScaler
scaler = MinMaxScaler()
X_num_scaled = scaler.fit_transform(X[num_escalar])
X_num_scaled_df = pd.DataFrame(X_num_scaled, columns=[f"{col}_mms" for col in num_escalar], index=X.index)

# OneHotEncoding
ohe = OneHotEncoder(drop='first', sparse_output=False, dtype=int)
X_ohe = ohe.fit_transform(X[var_ohe])
X_ohe_cols = [f"{var_ohe[i]}_{cat}" for i, cats in enumerate(ohe.categories_) for cat in cats[1:]]
X_ohe_df = pd.DataFrame(X_ohe, columns=X_ohe_cols, index=X.index)

# Compilar features
X_procesado = pd.concat([
    X_num_scaled_df,
    X[var_sin_transform],
    X_bin,
    X_ohe_df
], axis=1)

print(f"✓ Transformaciones aplicadas: {X_procesado.shape[1]} features")

✓ Transformaciones aplicadas: 70 features


## FASE 4 - Modelización (A_07)

In [8]:
# Usar top 26 features más importantes (basado en A_05)
# Simplificación: usar todos los features para mantener reproducibilidad
X_selected = X_procesado.copy()

# Configure CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

# RandomizedSearchCV
lr = LogisticRegression(max_iter=1000, random_state=42)
grid_search = RandomizedSearchCV(
    lr, param_grid, cv=skf, scoring='roc_auc', n_iter=30, 
    random_state=42, n_jobs=-1, verbose=0
)

grid_search.fit(X_selected, y)

print(f"✓ RandomizedSearchCV completado")
print(f"  Mejor CV AUC: {grid_search.best_score_:.4f}")
print(f"  Hiperparámetros: {grid_search.best_params_}")

✓ RandomizedSearchCV completado
  Mejor CV AUC: 0.8987
  Hiperparámetros: {'solver': 'lbfgs', 'penalty': 'l2', 'C': 10}


In [9]:
# Modelo final con hiperparámetros óptimos
modelo_final = LogisticRegression(
    C=grid_search.best_params_['C'],
    penalty='l2',
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

modelo_final.fit(X_selected, y)

y_pred_proba = modelo_final.predict_proba(X_selected)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

auc_score = roc_auc_score(y, y_pred_proba)
accuracy = accuracy_score(y, y_pred)
precision = precision_score(y, y_pred)
recall = recall_score(y, y_pred)
f1 = f1_score(y, y_pred)

print(f"✓ Modelo final entrenado")
print(f"\nMétricas:")
print(f"  AUC-ROC:  {auc_score:.4f}")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-Score: {f1:.4f}")
print(f"\n✅ Pipeline completado - Listo para A_09 (Pipelines)")

✓ Modelo final entrenado

Métricas:
  AUC-ROC:  0.9044
  Accuracy: 0.8264
  Precision: 0.7932
  Recall: 0.7212
  F1-Score: 0.7555

✅ Pipeline completado - Listo para A_09 (Pipelines)
